# GAN Training Postprocessing

This notebook generates samples from a trained model checkpoint (`.pt`) and performs an automated analysis on the generated realizations, reusing the functions defined in `visualize_and_generate.py` and `training_sample_plots.py`.

In [1]:
import os
import sys
from pathlib import Path
from IPython.display import Image, display

# Add the project root to the system path to allow module imports
notebook_dir = Path(os.getcwd())
if notebook_dir.name == 'GAN':
    project_root = str(notebook_dir.parent.parent)
elif notebook_dir.name == 'scripts':
    project_root = str(notebook_dir.parent)
else:
    project_root = str(notebook_dir)

if project_root not in sys.path:
    sys.path.append(project_root)

# Import necessary functions
from scripts.GAN.visualize_and_generate import plot_losses, generate_realizations
from scripts.Plots.training_sample_plots import (
    plot_cell_pdfs, 
    plot_realization_slices, 
    plot_entropy, 
    plot_3d_pyvista
)


c:\Users\mathi\miniconda3\envs\fluvgan\Lib\site-packages\h5py\__init__.py:36: UserWarning: h5py is running against HDF5 1.14.3 when it was built against 1.14.2, this may cause problems
  _warn(("h5py is running against HDF5 {0} when it was built against {1}, "
c:\Users\mathi\miniconda3\envs\fluvgan\Lib\site-packages\voxgan\models\base.py:46: DeprecationWarning: `TorchScript` support for functional optimizers is deprecated and will be removed in a future PyTorch release. Consider using the `torch.compile` optimizer instead.
  from torch.distributed.optim import ZeroRedundancyOptimizer


ModuleNotFoundError: No module named 'custom_plots'

In [ ]:
# === Configuration ===
SAVE_OUTPUTS = True  # Used to enable/disable saving behavior if needed

# Define the directory of the specific run
run_dir = r"C:\Users\mathi\Desktop\TU Delft\TU Delft year 5\Master Thesis\Thesis-project-DGM\outputs\RUN_2_1_epoch_batch_size_8_and_8"

# Define specific file paths (update these as needed)
csv_path = os.path.join(run_dir, "fluvgan_1_training_1_architecture_4_dcgan_no_one_hot_1_history.csv")
ckpt_path = os.path.join(run_dir, "architecture_4_dcgan_training_dataset_upper_plain_delta_no_one_hot_epochs_1_bs_8_run_1.pt")

# Set output directory
output_dir = os.path.join(run_dir, "postprocessing_results")
if SAVE_OUTPUTS:
    os.makedirs(output_dir, exist_ok=True)

# Number of realizations to generate for analysis
num_realizations = 10


## 1. Loss Visualization

In [ ]:
# Plot the losses from the CSV history
print(f"Visualizing losses from: {csv_path}")
if SAVE_OUTPUTS:
    plot_losses(csv_path, output_dir)

# Display the saved loss image inline
loss_img_path = os.path.join(output_dir, "loss_visualization.png")
if os.path.exists(loss_img_path):
    display(Image(filename=loss_img_path))


## 2. Generate Realizations

In [ ]:
# Generate realizations from the saved PyTorch checkpoint
print(f"Generating {num_realizations} realizations from: {ckpt_path}")
if SAVE_OUTPUTS:
    generate_realizations(ckpt_path, output_dir, num_realizations=num_realizations)


## 3. Analysis & Visualization

In [ ]:
print("Performing analysis on the generated samples...")

# Gather the generated realization files
generated_files = [f for f in os.listdir(output_dir) if f.startswith("realization_") and f.endswith(".npy")]

if generated_files and SAVE_OUTPUTS:
    print(f"Found {len(generated_files)} generated realizations for analysis.\n")
    
    # 1. Plot Cell PDFs
    try:
        plot_cell_pdfs(output_dir, output_dir, generated_files)
        pdf_img = os.path.join(output_dir, "first_20_cells_distributions.png")
        if os.path.exists(pdf_img): display(Image(filename=pdf_img))
    except Exception as e:
        print(f"Could not plot cell PDFs: {e}")
    
    # 2. Plot Realization Slices
    try:
        num_slice_samples = min(4, len(generated_files))
        plot_realization_slices(output_dir, output_dir, generated_files, num_files=num_slice_samples, x_slice=None, y_slice=None, z_slice=None)
        slice_img = os.path.join(output_dir, f"orthogonal_slices_{num_slice_samples}_samples.png")
        if os.path.exists(slice_img): display(Image(filename=slice_img))
    except Exception as e:
        print(f"Could not plot realization slices: {e}")
        
    # 3. Plot Entropy Matrices
    try:
        plot_entropy(output_dir, output_dir, generated_files, num_files=len(generated_files))
        dataset_name = os.path.basename(os.path.normpath(output_dir))
        entropy_img = os.path.join(output_dir, f"entropy_matrix_XY_{dataset_name}_{len(generated_files)}_samples.png")
        if os.path.exists(entropy_img): display(Image(filename=entropy_img))
    except Exception as e:
        print(f"Could not plot entropy matrices: {e}")
        
    # 4. Plot 3D PyVista
    try:
        plot_3d_pyvista(output_dir, output_dir, generated_files)
    except Exception as e:
        print(f"Could not plot 3D representation: {e}")
        
    print("\n✅ Analysis complete. Check the output directory for generated plots.")
else:
    print("No generated realizations found for analysis.")
